# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [5]:
# Setup
# pip install -r requirements.txt
import os, json
from google import genai
from google.genai import types

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env.example)"
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.1-flash-lite"


In [5]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [6]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain what a ticket classification system does in one sentence.",
    config=types.GenerateContentConfig(
        system_instruction="You are a helpful assistant."
    )
)

print("Response:")
print(response.text)

print("\nUsage:")
print(response.usage_metadata)


Response:
A ticket classification system automatically categorizes incoming support requests based on their content, allowing them to be routed to the appropriate team or prioritized according to their urgency.

Usage:
cache_tokens_details=None cached_content_token_count=None candidates_token_count=31 candidates_tokens_details=None prompt_token_count=19 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=19
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=50 traffic_type=None


In [7]:
prompt = "Suggest a short title for a customer support ticket about a user being unable to log in."

for temp in [0.1, 1.0]:
    print(f"\n{'=' * 50}")
    print(f"TEMPERATURE = {temp}")
    print(f"{'=' * 50}")

    for run in range(1, 4):
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction="You are a helpful assistant.",
                temperature=temp
            )
        )

        print(f"\nRun {run}:")
        print(response.text)


TEMPERATURE = 0.1

Run 1:
Here are a few options, depending on the tone you need:

**Direct & Professional:**
*   Unable to log in
*   Login issue
*   Authentication error

**Action-Oriented:**
*   Cannot access account
*   Login failure
*   Password/Login assistance needed

**Shortest:**
*   Login Error
*   Access Denied

**Recommendation:** **"Unable to log in"** is usually the best choice as it is clear, concise, and easy for support teams to categorize.

Run 2:
Here are a few options, depending on the tone you need:

**Direct & Professional:**
*   Unable to log in
*   Login issue
*   Authentication error

**Action-Oriented:**
*   Cannot access account
*   Login failure
*   Password/Login assistance needed

**Shortest:**
*   Login Error
*   Account Access

**Recommendation:** **"Unable to log in"** is usually the best choice as it is clear, concise, and easy for support teams to categorize.

Run 3:
Here are a few short title options, depending on the tone you need:

**Direct & Prof

**What changed, and why?**

For some reason there wasn't much difference between outputs, probably because there was a static system prompt and even the user prompt was well structured.
Since the instructions were as clear as they could be, the temperature didn't affect the overall 'sanity' of the answer.


## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [8]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

    # TODO: build the prompt for the given style ("zero_shot" | "few_shot" | "cot")
    # TODO: call the model and return the predicted label

# TODO: run all three styles over the tickets and assemble a results table


In [9]:
def classify(text, style):
    if style == "zero_shot":
        prompt = f"""
Classify the following support ticket into exactly one label:

Labels:
- billing
- bug
- feature_request
- other

Return ONLY the label.

Ticket:
{text}
"""

    elif style == "few_shot":
        prompt = f"""
Classify support tickets into one label.

Examples:

Ticket: "I was charged twice for my subscription."
Label: billing

Ticket: "The app crashes whenever I upload a photo."
Label: bug

Ticket: "Please add dark mode."
Label: feature_request

Ticket: "How do I change my profile picture?"
Label: other

Now classify this ticket.

Ticket:
{text}

Return ONLY the label.
"""

    elif style == "cot":
        prompt = f"""
You are classifying customer support tickets.

Think step by step about what category best fits the ticket.
Then return ONLY one of these labels:

billing
bug
feature_request
other

Ticket:
{text}
"""

    else:
        raise ValueError("Unknown prompt style")

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text.strip()

In [10]:
results = []

for ticket in tickets:
    row = {
        "ticket": ticket
    }

    for style in ["zero_shot", "few_shot", "cot"]:
        row[style] = classify(ticket, style)

    results.append(row)

In [12]:
import pandas as pd

df = pd.DataFrame(results)
print(df)

                                              ticket        zero_shot  \
0  {'id': 1, 'text': 'I was charged twice for my ...          billing   
1  {'id': 2, 'text': 'The export button throws a ...              bug   
2  {'id': 3, 'text': 'It would be great if you co...  feature_request   
3  {'id': 4, 'text': 'How do I reset my password?...            other   
4  {'id': 5, 'text': 'The app crashes on startup ...              bug   
5  {'id': 6, 'text': 'Can you send me a copy of m...          billing   
6  {'id': 7, 'text': 'Please add PDF export — CSV...  feature_request   
7  {'id': 8, 'text': 'Just wanted to say the new ...            other   

          few_shot              cot  
0          billing          billing  
1              bug              bug  
2  feature_request  feature_request  
3            other            other  
4              bug              bug  
5          billing          billing  
6  feature_request  feature_request  
7            other            other  


## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [18]:
import json

from google import genai
from google.genai import types
from openai import OpenAI

In [27]:
LABELS = ["billing", "bug", "feature_request", "other"]

PROMPT = """
Classify the following support ticket.

Possible labels:
- billing
- bug
- feature_request
- other

Return ONLY valid JSON in this format:

{{
    "label": "...",
    "confidence": 0.0,
    "reason": "..."
}}

Ticket:
{text}
"""

In [20]:
def validate_result(data):
    if data["label"] not in LABELS:
        raise ValueError(f"Invalid label: {data['label']}")

    if not isinstance(data["confidence"], (int, float)):
        raise ValueError("Confidence must be numeric")

    if not (0 <= data["confidence"] <= 1):
        raise ValueError("Confidence must be between 0 and 1")

    return data

In [21]:
def classify_structured(text):
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=PROMPT.format(text=text),
            config=types.GenerateContentConfig(
                temperature=0.1,
                response_mime_type="application/json"
            )
        )

        data = json.loads(response.text)

        return validate_result(data)

    except (json.JSONDecodeError, KeyError, ValueError) as e:
        print("Gemini error:", e)
        return None

In [28]:
print("===== GEMINI RESULTS =====\n")

for ticket in tickets:
    print(ticket)
    print(classify_structured(ticket))
    print("-" * 60)

===== GEMINI RESULTS =====

{'id': 1, 'text': 'I was charged twice for my subscription this month. Please refund the extra charge.'}
{'label': 'billing', 'confidence': 1.0, 'reason': 'The user is reporting a duplicate charge and requesting a refund, which directly relates to billing and payment issues.'}
------------------------------------------------------------
{'id': 2, 'text': 'The export button throws a 500 error every time I click it on the reports page.'}
{'label': 'bug', 'confidence': 1.0, 'reason': 'The user is reporting a 500 error, which indicates a server-side failure or functional defect in the application.'}
------------------------------------------------------------
{'id': 3, 'text': 'It would be great if you could add a dark mode to the dashboard.'}
{'label': 'feature_request', 'confidence': 1.0, 'reason': 'The user is explicitly suggesting the addition of a new functionality (dark mode) to the dashboard.'}
------------------------------------------------------------


In [29]:
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [30]:
def classify_structured_ollama(text):
    try:
        response = ollama.chat.completions.create(
            model="llama3.2:3b",
            temperature=0.1,
            messages=[
                {
                    "role": "user",
                    "content": PROMPT.format(text=text)
                }
            ]
        )

        data = json.loads(response.choices[0].message.content)

        return validate_result(data)

    except (json.JSONDecodeError, KeyError, ValueError) as e:
        print("Ollama error:", e)
        return None

In [31]:
print("===== OLLAMA RESULTS =====\n")

for ticket in tickets:
    print(ticket)
    print(classify_structured_ollama(ticket))
    print("-" * 60)

===== OLLAMA RESULTS =====

{'id': 1, 'text': 'I was charged twice for my subscription this month. Please refund the extra charge.'}
{'label': 'billing', 'confidence': 0.9, 'reason': 'The ticket text explicitly mentions a billing issue and requests a refund.'}
------------------------------------------------------------
{'id': 2, 'text': 'The export button throws a 500 error every time I click it on the reports page.'}
{'label': 'bug', 'confidence': 0.9, 'reason': 'Error when clicking export button on reports page'}
------------------------------------------------------------
{'id': 3, 'text': 'It would be great if you could add a dark mode to the dashboard.'}
{'label': 'feature_request', 'confidence': 1.0, 'reason': 'The ticket text explicitly requests a feature (dark mode) for the dashboard.'}
------------------------------------------------------------
{'id': 4, 'text': "How do I reset my password? I can't find the link anywhere."}
{'label': 'other', 'confidence': 0.0, 'reason': 'Th

In [32]:
print("===== COMPARISON =====\n")

for ticket in tickets:
    print(ticket)

    print("\nGemini")
    print(classify_structured(ticket))

    print("\nOllama")
    print(classify_structured_ollama(ticket))

    print("=" * 70)

===== COMPARISON =====

{'id': 1, 'text': 'I was charged twice for my subscription this month. Please refund the extra charge.'}

Gemini
{'label': 'billing', 'confidence': 1.0, 'reason': 'The user is reporting a duplicate charge and requesting a refund, which directly relates to billing and payment issues.'}

Ollama
{'label': 'billing', 'confidence': 0.9, 'reason': 'The ticket text explicitly mentions a billing issue and requests a refund.'}
{'id': 2, 'text': 'The export button throws a 500 error every time I click it on the reports page.'}

Gemini
{'label': 'bug', 'confidence': 1.0, 'reason': 'The user is reporting a 500 error, which indicates a server-side failure or functional defect in the application.'}

Ollama
{'label': 'bug', 'confidence': 0.9, 'reason': 'Error occurs when clicking export button on reports page'}
{'id': 3, 'text': 'It would be great if you could add a dark mode to the dashboard.'}

Gemini
{'label': 'feature_request', 'confidence': 1.0, 'reason': 'The user is exp

In [34]:
#def classify_structured(text):
    # TODO: use the API's JSON / structured-output mode at low temperature
    # TODO: parse the JSON, validate label in LABELS and 0 <= confidence <= 1
    # TODO: handle at least one malformed-output case gracefully
    #raise NotImplementedError

# TODO: run the SAME structured prompt against local Ollama (OpenAI-compatible
# endpoint at http://localhost:11434/v1) and note how reliably it returns valid JSON.


**Local vs hosted: did the small local model produce valid JSON as reliably?**

In my tests, both the hosted Gemini model and the local Ollama model consistently returned valid JSON. However, while the JSON format was reliable in both cases, the local model made more classification mistakes and produced less consistent confidence scores than Gemini.
